# Dynamic Session Test Notebook

This notebook mirrors `static_session_test.ipynb` but evaluates **dynamic features** using real session stroke data.

In [1]:
# 1. Import Required Libraries
import os
import sys
import json
import time
import numpy as np
import pandas as pd
from datetime import datetime, timezone

# Import dynamic feature module from local features/ folder
FEATURES_DIR = os.path.join(os.getcwd(), 'features')
if FEATURES_DIR not in sys.path:
    sys.path.insert(0, FEATURES_DIR)

from dynamic_features import DynamicFeatureExtractor, DrawingSessionValidator, StrokeAnalyzer

print('✅ Imports successful')

✅ Imports successful


In [2]:
# 2. Load Session Data (same session style used by static_session_test)
session_json_path = os.path.join('Records', 'session-1772430903272', 'JSON', 'session-1772430903272.json')

with open(session_json_path, 'r', encoding='utf-8') as f:
    raw_session = json.load(f)

task_keys = sorted(raw_session.get('tasks', {}).keys(), key=lambda k: int(k))
print(f"✅ Session loaded: {raw_session.get('sessionId')}")
print(f"Tasks found: {task_keys}")

✅ Session loaded: session-1772430903272
Tasks found: ['1', '2', '3']


In [3]:
# 3. Build Dynamic-Extractor-Compatible Task Sessions
def to_iso_utc(ms):
    return datetime.fromtimestamp(ms / 1000, tz=timezone.utc).isoformat().replace('+00:00', 'Z')

def build_task_dynamic_session(raw_session_obj, task_key):
    task = raw_session_obj['tasks'][task_key]
    task_strokes = task.get('strokes', [])

    if not task_strokes:
        return None

    min_ts = float('inf')
    max_ts = float('-inf')
    normalized_strokes = []

    for idx, stroke in enumerate(task_strokes):
        points = stroke.get('points', [])
        if not points:
            continue

        point_timestamps = [p.get('timestamp') for p in points if p.get('timestamp') is not None]
        if not point_timestamps:
            continue

        stroke_start_ms = min(point_timestamps)
        stroke_end_ms = max(point_timestamps)

        min_ts = min(min_ts, stroke_start_ms)
        max_ts = max(max_ts, stroke_end_ms)

        normalized_strokes.append({
            'strokeId': stroke.get('strokeId', f'stroke-{idx+1}'),
            'startTime': stroke.get('startTime') or to_iso_utc(stroke_start_ms),
            'endTime': stroke.get('endTime') or to_iso_utc(stroke_end_ms),
            'points': points
        })

    if not normalized_strokes:
        return None

    dynamic_session = {
        'sessionId': f"{raw_session_obj.get('sessionId')}-task-{task_key}",
        'sessionStartTime': to_iso_utc(min_ts),
        'sessionEndTime': to_iso_utc(max_ts),
        'strokes': normalized_strokes,
        'deviceInfo': {'type': 'unknown', 'model': 'unknown'},
        'taskMeta': {
            'task_key': task_key,
            'task_title': task.get('title', f'Task {task_key}'),
            'task_type': task.get('type', 'unknown')
        }
    }

    return dynamic_session

task_sessions = []
for key in task_keys:
    built = build_task_dynamic_session(raw_session, key)
    if built is not None:
        task_sessions.append(built)

print(f"✅ Built {len(task_sessions)} task-level dynamic sessions")
for ts in task_sessions:
    print(f"  - {ts['taskMeta']['task_key']}: {ts['taskMeta']['task_title']} (strokes={len(ts['strokes'])})")

✅ Built 3 task-level dynamic sessions
  - 1: Two-Pentagon Copy (strokes=10)
  - 2: House Drawing Copy (strokes=34)
  - 3: Clock Drawing (strokes=20)


In [4]:
# 4. Validate and Extract Dynamic Features per Task
rows = []

for task_session in task_sessions:
    validator_result = DrawingSessionValidator.validate_session(task_session)

    extractor = DynamicFeatureExtractor(task_session, validate=True)
    features = extractor.extract_all_features()

    rows.append({
        'task_key': task_session['taskMeta']['task_key'],
        'task_title': task_session['taskMeta']['task_title'],
        'validation_ok': validator_result.is_valid,
        'validation_errors': len(validator_result.errors),
        'total_strokes': features.total_strokes,
        'total_points': features.total_points,
        'avg_speed': features.avg_speed,
        'max_speed': features.max_speed,
        'speed_variance': features.speed_variance,
        'avg_acceleration': features.avg_acceleration,
        'max_acceleration': features.max_acceleration,
        'acceleration_variance': features.acceleration_variance,
        'total_drawing_time_ms': features.total_drawing_time,
        'total_pause_time_ms': features.total_pause_time,
        'number_of_pauses': features.number_of_pauses,
        'avg_pause_duration_ms': features.avg_pause_duration,
        'stroke_frequency_hz': features.stroke_frequency,
        'direction_changes': features.direction_changes,
        'avg_curvature': features.avg_curvature,
        'max_curvature': features.max_curvature,
        'drawing_area': features.drawing_area
    })

dynamic_df = pd.DataFrame(rows).sort_values('task_key').reset_index(drop=True)
print('Dynamic feature summary by task:')
display(dynamic_df.head(len(dynamic_df)))

Dynamic feature summary by task:


,task_key,task_title,validation_ok,validation_errors,total_strokes,total_points,avg_speed,max_speed,speed_variance,avg_acceleration,...,acceleration_variance,total_drawing_time_ms,total_pause_time_ms,number_of_pauses,avg_pause_duration_ms,stroke_frequency_hz,direction_changes,avg_curvature,max_curvature,drawing_area
0,1,Two-Pentagon Copy,True,0,10,563,0.459065,2.270355,0.155726,0.034351,...,0.002654,7808.0,3534.0,9,392.666667,0.892857,87,0.222963,3.141593,154784.25
1,2,House Drawing Copy,True,0,34,2114,0.345905,3.000000,0.107974,0.024124,...,0.001491,29518.0,23341.0,33,707.303030,0.645125,185,0.140821,4.188790,211717.00
2,3,Clock Drawing,True,0,20,1229,0.325439,2.501389,0.082422,0.023990,...,0.001397,13259.0,9896.0,19,520.842105,0.863819,232,0.301106,6.283185,181095.75


In [ ]:
# 5. Benchmark extraction time (similar spirit to static benchmark cell)
RUNS = 5
timing_rows = []

for task_session in task_sessions:
    task_name = task_session['taskMeta']['task_title']
    for run_id in range(1, RUNS + 1):
        t0 = time.perf_counter()
        extractor = DynamicFeatureExtractor(task_session, validate=True)
        _ = extractor.extract_all_features()
        elapsed_ms = (time.perf_counter() - t0) * 1000

        timing_rows.append({
            'task_title': task_name,
            'run': run_id,
            'extraction_ms': elapsed_ms
        })

timing_df = pd.DataFrame(timing_rows)
timing_summary = (
    timing_df.groupby('task_title', as_index=False)
    .agg(
        mean_ms=('extraction_ms', 'mean'),
        min_ms=('extraction_ms', 'min'),
        max_ms=('extraction_ms', 'max')
    )
)

print('Dynamic extraction timing summary (ms):')
display(timing_summary.sort_values('task_title'))

ValueError: Length mismatch: Expected axis has 5 elements, new values have 4 elements

In [ ]:
# 6. Optional: inspect low-level stroke metrics for one task
sample_task = task_sessions[0]
sample_stroke = sample_task['strokes'][0]

distances = StrokeAnalyzer.calculate_distances(sample_stroke)
time_deltas = StrokeAnalyzer.calculate_time_deltas(sample_stroke)
velocities, avg_vel, max_vel, min_vel = StrokeAnalyzer.calculate_velocity(sample_stroke)
accels, avg_accel, max_accel = StrokeAnalyzer.calculate_acceleration(sample_stroke)
curvatures, avg_curv, max_curv = StrokeAnalyzer.calculate_curvature(sample_stroke)
dir_changes = StrokeAnalyzer.count_direction_changes(sample_stroke, angle_threshold=30)

print(f"Task: {sample_task['taskMeta']['task_title']}")
print(f"Stroke ID: {sample_stroke['strokeId']}")
print(f"Distances count: {len(distances)}")
print(f"Time deltas count: {len(time_deltas)}")
print(f"Avg velocity: {avg_vel:.6f} px/ms | Max velocity: {max_vel:.6f} px/ms")
print(f"Avg acceleration: {avg_accel:.8f} px/ms² | Max acceleration: {max_accel:.8f} px/ms²")
print(f"Avg curvature: {avg_curv:.8f} | Max curvature: {max_curv:.8f}")
print(f"Direction changes (>30°): {dir_changes}")

In [ ]:
# 7. Save outputs (parallel to static notebook CSV outputs)
dynamic_df.to_csv('dynamic_feature_summary.csv', index=False)
timing_df.to_csv('dynamic_feature_benchmarks_raw.csv', index=False)
timing_summary.to_csv('dynamic_feature_benchmarks.csv', index=False)

print('✅ Saved files:')
print('  - dynamic_feature_summary.csv')
print('  - dynamic_feature_benchmarks_raw.csv')
print('  - dynamic_feature_benchmarks.csv')

## Notes
- This notebook uses the same session source style as `static_session_test.ipynb`.
- It reconstructs dynamic-session fields required by `DynamicFeatureExtractor`.
- You can switch `session_json_path` to any other record JSON for re-testing.